### **Figure 4.** Bar plots and topographies of relevant parameters (for gradiometers). 

In [ ]:
"""
Script for Generating Source Data for Figure 4.

This notebook processes LME statistics, mean sensor values, and individual 
spectral parameter values to export as TSV files needed for barplots and 
topographic plots of Figure 4. These files will be read by figure04_script.ipynb 
to create the final figure.

Author: Maité Crespo García
Affiliation: MRC Cognition and Brain Sciences Unit, Cambridge, UK
Date: 2026 (last modified)
"""

import pandas as pd
import numpy as np
import mne
import os

print(os.getcwd())
datadir = os.path.abspath(os.path.join(os.path.dirname( os.getcwd() ), '.', 'data'))
print(f'Data directory: {datadir}') # Data dir of this repository (the output .tsv files are already saved in this directory, so careful if you want to run this notebook again, to avoid overwriting them)

maindir = '' # main directory where the bids repository is located, with the derivatives folder inside it.

pipver = ''
task = 'rest'
phases = ['p2', 'p5']

lfreq = 0.1 #Hz
hfreq = 145.0 #Hz
fsample = 300.0 #Hz
frange = f"{round(lfreq, 1)}-{int(hfreq)}Hz"

trans = True # Whether to use head transformation or not
zmm = 44 # destination z coordinate head position in mm

icselection = 'ecg04eog08' # 'allbutecg04' #'eog08' #
proc = 'filt' + icselection #'sss' #'clean'

# ---- File with subjects and arms ----
subjlistfile = os.path.join(datadir,f'meglong_{task}_subjects.tsv')

# ---- File with subjects and age in each phase ----
agefile = os.path.join(datadir,f'meglong_rest_age.tsv')

megtype = 'grad' # 'grad', 'mag',
age_groups = ['Young', 'Middle', 'Old']

# --- Read in the subjects and age dataframes ---
subjectsdf = pd.read_csv(subjlistfile, sep='\t', index_col=0)
agedf = pd.read_csv(agefile, sep='\t', index_col=0)
agedf.rename(columns={'p2_meg_age': 'p2_age', 'p5_meg_age': 'p5_age'}, inplace=True)

subjects = subjectsdf.index.tolist()
agedf = agedf.loc[subjects]
subjectsdf = subjectsdf.loc[subjects]

# --- Define age groups based on age in phase 2
age_bins = np.percentile(agedf['p2_age'], [0, 100/3, 2*100/3, 100])
subjectsdf['Age_group'] = pd.cut(agedf['p2_age'], bins=age_bins, labels=['Young', 'Middle', 'Old'], include_lowest=True)
agedf.loc[subjects, 'Age_group'] = subjectsdf.loc[subjects, 'Age_group']

# Generate structural summary metrics without using loops
summary_df = agedf.groupby('Age_group', observed=True).agg(
    min_age=('p2_age', 'min'),
    max_age=('p2_age', 'max'),
    min_age_p5=('p5_age', 'min'),
    max_age_p5=('p5_age', 'max'),
    n_subjects=('p2_age', 'count')
)

# Convert directly to a dictionary matching your original schema
age_groups_dict = summary_df.to_dict(orient='index')

print(age_groups_dict)

phases_dict = {
    'p2': {'label': 'Phase 2', 'color': 'royalblue'},
    'p5': {'label': 'Phase 5', 'color': 'orange'}
}

{'Young': {'min_age': 24.3, 'max_age': 47.6, 'min_age_p5': 35.6, 'max_age_p5': 58.7, 'n_subjects': 45}, 'Middle': {'min_age': 47.8, 'max_age': 63.1, 'min_age_p5': 58.9, 'max_age_p5': 74.5, 'n_subjects': 44}, 'Old': {'min_age': 63.2, 'max_age': 84.1, 'min_age_p5': 70.9, 'max_age_p5': 95.5, 'n_subjects': 44}}


### **Figure 4**: bar plots with parameter means, LME estatistics, and topographies of power (3 columns)

In [ ]:
# Figure 4: Barplots of aperiodic and band parameters and topographies (full page, 3 columns - topographies with power only)
fitting_param = 'finley'

bands = ['theta', 'alpha', 'beta'] #, 'gamma'
parameters = ['exponent']
for band in bands:
    if band not in ['gamma']:
        parameters.append(f'{band}_band_power')
    if band not in ['theta']:
        parameters.append(f'{band}_peak_freq')


# Prepare df for barplots
# Directories and file names
deriv_folder = f'aperiodic_filt{frange}_fs{int(fsample)}Hz_trans_z{zmm}mm'
taskref = 'rest'
phaseref = 'p5'
armref = 1
bids_project_folder = f'BIDS_long_{phaseref}_{taskref}_arm{armref}'
deriv_root = os.path.join(maindir, bids_project_folder,
                          'derivatives', deriv_folder)
statsdir = os.path.join(deriv_root, 'stats')
stats_folder = 'lme_maxT_finley_2betas_10000rand'
loaddir = os.path.join(statsdir, stats_folder)

# This TSV file contains the spectral parameters for each subject and phase, in 
# addition to age and lag. It was primarily used for the LME analyses, but it contains 
# the mean values for each parameter, averaged across channels, which is what we 
# need to compute the phase and age group mean for the bar plots.
varfile = os.path.join(loaddir, f'aperiodic_stier_{proc}_{fitting_param}_2betas_allvars_means.tsv')

# Load file with parameters
df_vars = pd.read_csv(varfile, sep='\t')    
df_vars['phase'] = df_vars['subject_phase'].str.split('_').str[1]
df_vars['subject'] = df_vars['subject_phase'].str.split('_').str[0]

# Define colors for each age group
age_group_colors = {
    'Young': 'dimgray',
    'Middle': 'dodgerblue',
    'Old': 'darkorange'
}

# ---- FUNCTIONS for reading the data ----
def get_varmean(datafile, megtype):
# Function to get mean values for each variable and channel, averaged across phases for each subject.
    df = pd.read_csv(datafile, sep='\t').set_index(['row'])
    df = df.drop(columns=['task', 'Age0', 'deltaAge'])
    df2 = df[df.phase == 'p2']
    df2 = df2.set_index('subject')
    df2.drop(columns=['phase'], inplace=True)
    df5 = df[df.phase == 'p5']
    df5 = df5.set_index('subject')
    df5.drop(columns=['phase'], inplace=True)
    df = pd.concat([df2, df5]).groupby('subject').mean()
    if megtype == 'grad':
        channels = [c[:-1] + '1' for c in df.columns if c.startswith('MEG') and (c.endswith('2'))]
        tmpdf = pd.DataFrame(columns=channels)
        for c in channels:
            tmpdf[c] = df[[f'{c[:-1]}2', f'{c[:-1]}3']].mean(axis=1)
        df = pd.concat([df, tmpdf], axis=1)
        df = df.drop(columns=[c for c in df.columns if c.endswith('2') or c.endswith('3')])
    else:
        channels = [c for c in df.columns if c.startswith('MEG') and (c.endswith('1'))]

    count = df.count()
    mean = df.mean()
    return mean, count, channels


def get_channels_positions(channels):
    # Function to get channel positions from layout file.

    # Load standard positions for the channels
    path_to_fieldtrip = '' # path to fieldtrip toolbox, which contains the layout files with the channel positions
    layoutdir = os.path.join(path_to_fieldtrip, 'template', 'layout')
    layout = mne.channels.read_layout(os.path.join(layoutdir, 'neuromag306mag.lay'))# , scale = True

    # --- Preparation for topographic plots --- 
    # get the positions of the channels
    pos = []                        
    for ch in channels:
        if ch in layout.names:
            pos.append(layout.pos[layout.names.index(ch),0:2]/5)
        else:
            raise ValueError(f'Channel {ch} not found in layout')

    pos = np.array(pos) 
    return pos

# --- End of functions ----

# File with the statistics for each spectral parameter and age effect, to be used
# for the annotations of the bar plots (T values, p values and significance stars)
statsfile = os.path.join(maindir, 'Control_analyses_Results', f'CA_all_control_analyses_combined.csv')
statsdf = pd.read_csv(statsfile, index_col=0)
statsdf = statsdf.loc[:,'Effect':'Beta']

pos = None
df_stats_list = []
df_values_list = []

# Loop over parameters and create bar plots and topographies
for i, parameter in enumerate(parameters):

    parameter_stats = statsdf[statsdf['Parameter'] == f'{parameter.replace('_', ' ').title()} ({megtype})']

    df_stats_list.append(parameter_stats)

    col_y = f'{parameter}_{megtype}'

    df_values_list.append(df_vars[['subject', 'phase', col_y]].copy().set_index(['subject', 'phase']))

    df_barplotdata_list = []

    # Loop over age groups to calculate mean and SEM for each phase
    for age_group in age_groups:

        # Create list with subjects in the age group
        group_subjects = agedf[agedf['Age_group'] == age_group].index.tolist()

        # Create a temporary DataFrame for the age group and sort it by subject and phase
        df_agegroup = df_vars[df_vars['subject'].isin(group_subjects)].copy()
        df_agegroup.sort_values(by=['subject', 'phase'], inplace=True)

        # Create a temporary variable with the difference between phases for each subject and calculate the SEM of the difference (to be used as error bars in the plot)
        sem_val = df_agegroup.groupby(['subject'])[col_y].diff().dropna().sem()

        # Loop over phases to calculate mean for each phase
        for phase in phases:
            # Create a temporary DataFrame for the age group and phase, and calculate the mean value for the variable of interest            
            df_subset = df_vars[(df_vars['subject'].isin(group_subjects)) & (df_vars['phase'] == phase)]
            mean_value = df_subset[col_y].mean()

            tmp_dict = {'Age_group': age_group, 'Phase': phase, 'Mean': mean_value, 'SEM': sem_val, 'Color': age_group_colors[age_group], 'Alpha': 0.4 if phase=='p2' else 1.0, 'min_age': str(age_groups_dict[age_group]['min_age']) if phase=='p2' else str(age_groups_dict[age_group]['min_age_p5']), 'max_age': str(age_groups_dict[age_group]['max_age']) if phase=='p2' else str(age_groups_dict[age_group]['max_age_p5']), 'n_subjects': str(age_groups_dict[age_group]['n_subjects'])}

            df_barplotdata_list.append(tmp_dict)

    # --- End of loop over age groups and phases, df_data is now ready for plotting

    df_barplotdata = pd.DataFrame(df_barplotdata_list)
    df_barplotdata.to_csv(os.path.join(datadir, f'figure04_{parameter}_{megtype}_barplot.tsv'), sep='\t', index=False)
    
    if 'exponent' in parameter or 'power' in parameter:
        # This TSV file contains the values of a spectral parameter, for a sensor 
        # type, for all the subjects, phases, and channels. We will use it to compute 
        # the mean value for each channel, averaged across subjects and phases, 
        # to plot the topography of each parameter.
        datafile = os.path.join(
            statsdir, f'aperiodic_stier_{proc}_{fitting_param}_{megtype}{parameter}_2betas.tsv'
        )     
        varmean, _, channels = get_varmean(datafile, megtype)

        if pos is None: # read channel positions only once, since they are the same for all the topographies
            pos = get_channels_positions(channels)
            pos -= 0.1
            pos =  (pos*1.2)
            pos[:,1] = pos[:,1] + 0.014 
            pos[:,0] = pos[:,0] + 0.007

        topovals = varmean[channels].to_numpy(float)

        # Create a DataFrame with the channel names, positions, and mean values for the parameter, and save it as a TSV file to be used for the topographic plots in the figure 4 legend (with power/exponent values for each channel)
        df_topo = pd.DataFrame({'Channel': channels, 'Position_x': pos[:,0], 'Position_y': pos[:,1] , 'Value': topovals})

        df_topo.to_csv(os.path.join(datadir, f'figure04_{parameter}_{megtype}_topo.tsv'), sep='\t', index=False)

df_values = pd.concat(df_values_list, axis=1).reset_index()
df_values.to_csv(os.path.join(datadir, f'figure04_{megtype}_indiv_values.tsv'), sep='\t', index=False)
        
# --- Concatenate all the dataframes with the statistics for each parameter and save it as a TSV file, to be used for the annotations of the bar plots in the figure 4 legend (T values, p values and significance stars) ---
df_stats = pd.concat(df_stats_list)
df_stats.to_csv(os.path.join(datadir, f'figure04_{megtype}_stats.tsv'), sep='\t', index=False)